# SSR na ZedBoard / PYNQ — klasyfikacja plików `.mem`

Ładuje overlay, definiuje funkcje pomocnicze, a następnie odpala pojedynczo
wybrane pliki `synth_*.mem`.

## 1. Setup — uruchom raz
Ładuje overlay (`ssr.bit` + `ssr.hwh`), wypisuje używane pliki i definiuje funkcje.

In [ ]:
import numpy as np
from pynq import Overlay, allocate
import time, os

ol  = Overlay("ssr.bit")
dma = ol.axi_dma_0
try:
    ssr = ol.top_ssr_0
except AttributeError:
    from pynq import MMIO
    ssr = MMIO(0x43C00000, 0x10000)

CTRL, STATUS, RESULT = 0x00, 0x04, 0x08
N_SAMPLES = 16000
LABEL = {0: "OTHER", 1: "ON", 2: "OFF"}

# --- pliki audio uzywane w tescie (klasa -> plik) ---
files = {
    "on":    "synth_on.mem",     # oczekiwane: 1 (ON)
    "off":   "synth_off.mem",    # oczekiwane: 2 (OFF)
    "other": "synth_other.mem",  # oczekiwane: 0 (OTHER)
}

def load_mem(path):
    vals = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            v = int(line, 16) & 0xFFFF
            if v & 0x8000:
                v -= 0x10000
            vals.append(v)
    return np.array(vals, dtype=np.int16)

buf = allocate(shape=(N_SAMPLES,), dtype=np.int16)

def classify_file(path, verbose=True):
    """Klasyfikuje jeden konkretny plik .mem (po sciezce)."""
    data = load_mem(path)
    n = min(len(data), N_SAMPLES)

    ssr.write(CTRL, 1)
    time.sleep(0.001)
    buf[:n] = data[:n]
    if n < N_SAMPLES:
        buf[n:] = 0

    dma.sendchannel.transfer(buf)
    dma.sendchannel.wait()

    t0 = time.time()
    while not (ssr.read(STATUS) & 0x1):
        if time.time() - t0 > 10.0:
            print(f"  TIMEOUT: {path}"); return None
    res = ssr.read(RESULT) & 0x3
    if verbose:
        print(f"{os.path.basename(path):20s} -> {res} ({LABEL[res]})")
    return res

print("Pliki audio do testu:")
for name, path in files.items():
    print(f"  {name:6s} -> {path}")
print("\nReady. classify_file(path).")

## 2. Klasyfikacja pojedynczych plików
Każda komórka odpala jeden plik.

In [ ]:
classify_file(files["on"])     # oczekiwane: 1 (ON)

In [ ]:
classify_file(files["off"])    # oczekiwane: 2 (OFF)

In [ ]:
classify_file(files["other"])  # oczekiwane: 0 (OTHER)